# Base Model Validation (no fine-tuning)

Run the bare base model `INSAIT-Institute/BgGPT-Gemma-3-4B-IT` on the same Bulgarian mental-health test battery and score it with the 3-judge panel. This is the **before** baseline every fine-tune is compared against.

**Sequence**
1. Setup
2. Load base model
3. Run test prompts (in-domain, out-of-domain, edge cases)
4. Judge with the Mistral / Llama-Guard-4 / Nemotron panel
5. Log responses + judge scores to MLflow
6. Publish the executed notebook


In [ ]:
# ###### Colab bootstrap ######
# On Colab the [experiments] extras pulls both requirements_package.txt and
# requirements_experiments.txt in one pip command — single source of truth lives
# in setup.py's extras_require. Then bootstrap() loads Colab Secrets into
# os.environ and brings up Tailscale so *.ts.net hostnames are reachable.
# Locally bootstrap() just loads .env and checks the tailnet — no installs.
#
# Required Colab Secrets (key icon → Add new secret → toggle "Notebook access"):
#   TAILSCALE_AUTHKEY   – from https://login.tailscale.com/admin/settings/keys
#   MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME, MLFLOW_TRACKING_INSECURE_TLS
#   MINIO_ENDPOINT / MINIO_ACCESS_KEY / MINIO_SECRET_KEY / MINIO_SECURE
#   HF_TOKEN
import os, subprocess, sys
IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "burnit_bg[experiments] @ git+https://github.com/kirilyotov/BurnIT-BG.git",
    ])

from utils.colab import bootstrap
bootstrap()


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device: {props.name}")
    print(f"VRAM:   {props.total_memory / 1024**3:.2f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")



## 1. Setup & config

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'data_platform').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from data_platform.common import set_env
from data_platform.storage import MinioStorage

from experiments.shared.mlflow_utils import setup_run, log_responses, stage, log_training_curves
from experiments.shared.judge_utils import judge_and_log, JudgePanel
from experiments.shared.mlflow_judges import get_scorers, batteries_to_eval_dataset, reset_cache as reset_judge_cache
from experiments.shared.prompt_registry import register_burnit_prompts
from experiments.shared.inference_utils import (
    run_full_test_battery,
    TEST_PROMPTS_IN_DOMAIN, TEST_PROMPTS_OUT_OF_DOMAIN, TEST_PROMPTS_EDGE,
    format_prompt,
)
from experiments.shared.model_utils import (
    DEFAULT_MODEL_NAME, GEMMA3_MODEL, load_model_unsloth, apply_qlora, apply_dora, apply_lora,
    count_trainable_params, cuda_device_info, default_repo_for, push_to_hf,
)
from experiments.shared.eval_utils import (
    compute_perplexity, benchmark_speed, vram_snapshot,
    compute_bertscore, compute_rouge,
    plot_train_eval_loss, plot_grad_norm, plot_metric_scorecard, overfit_summary,
)
from experiments.shared.notebook_utils import log_executed_notebook
from experiments.shared.dataset_utils import (
    load_alpaca_dataset, dataset_statistics, alpaca_to_prompt,
)
from experiments.shared.dataset_browser import list_datasets, pick_dataset, download_dataset, resolve
from experiments.shared.dataset_loader import get_dataset, LoadedDataset
from transformers import EarlyStoppingCallback

In [ ]:
MODEL_NAME          = os.getenv("MODEL_NAME",       GEMMA3_MODEL)

In [ ]:
set_env(quiet=True)
# Throttle mlflow.genai.evaluate's scorer thread pool BEFORE the panel runs.
# Default is 4 parallel scorers; with our 3-call panel that'd be 12 concurrent
# requests to a 40 RPM NVIDIA endpoint — instant 429 storm. We use 2 workers
# and let the per-model RpmBucket in NvidiaChatClient pace the wire.
os.environ.setdefault("MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS", "2")
# Optional per-process RPM ceiling (per NVIDIA model key). Free tier is ~40
# RPM per key; we default to 30 to leave headroom for retries. Override via
# the NVIDIA_DEFAULT_RPM env var in Colab Secrets if you have a higher quota.
os.environ.setdefault("NVIDIA_DEFAULT_RPM", "30")

tracking, tags, run_name = setup_run(
    experiment='base_validation',
    model=MODEL_NAME,
    stage="experiment",
    mlflow_experiment_name='burnit-bg-experiments',
)
print(f"run_name = {run_name}")
print(f"tags     = {tags}")
print(f"machine  = {cuda_device_info()}")


## 2. Load base model

In [ ]:
HF_TOKEN = os.getenv("HF_TOKEN")
with stage(tracking, "load_model"):
    model, tokenizer = load_model_unsloth(
        MODEL_NAME, max_seq_length=2048, load_in_4bit=True, token=HF_TOKEN,
    )
    print(count_trainable_params(model))


## 3-5. Inference + judge + log to MLflow

In [ ]:
# All judge/log metrics go to ONE MLflow run.
import mlflow
from contextlib import nullcontext

_ctx = nullcontext() if mlflow.active_run() else tracking.run(run_name=run_name, tags=tags)
with _ctx:
    _run = mlflow.active_run()
    _run_id = _run.info.run_id if _run else None
    _exp_id = _run.info.experiment_id if _run else None
    _uri = os.getenv("MLFLOW_TRACKING_URI", "").rstrip("/")
    if _run_id and _uri:
        print(f"\n>>> MLflow run: {_uri}/#/experiments/{_exp_id}/runs/{_run_id}")
        print(f"    run_id={_run_id}  experiment_id={_exp_id}\n")

    tracking.log_params({"method": "base_only", "base_model": MODEL_NAME, "fine_tuning": False})

    with stage(tracking, "inference_test"):
        # Register the test battery as a Dataset so the prompt set
        # shows up in MLflow's Datasets tab (no training data here).
        # Self-imports the prompt constants so this works even when the
        # notebook's import cell didn't bring them into scope.
        try:
            import pandas as _pd, mlflow as _mlf, mlflow.data as _mld
            from experiments.shared.inference_utils import (
                TEST_PROMPTS_IN_DOMAIN as _IN,
                TEST_PROMPTS_OUT_OF_DOMAIN as _OOD,
                TEST_PROMPTS_EDGE as _EDGE,
            )
            _battery_records = (
                [{"battery": "in_domain", "prompt": q} for q in _IN]
                + [{"battery": "out_of_domain", "prompt": q} for q in _OOD]
                + [{"battery": "edge", "prompt": q} for q in _EDGE]
            )
            _battery_ds = _mld.from_pandas(_pd.DataFrame(_battery_records),
                                           name="burnit-bg-test-battery")
            _mlf.log_input(_battery_ds, context="test")
            print(f"[mlflow] test battery registered ({len(_battery_records)} prompts)")
        except Exception as _ds_exc:
            print(f"[mlflow] test battery register skipped: {_ds_exc}")
        batteries = run_full_test_battery((model, tokenizer), max_new_tokens=256)
        log_responses(tracking, experiment="base_validation",
                      model_checkpoint=MODEL_NAME, **batteries)
        for k, v in batteries.items():
            print(f"-- {k} --")
            for entry in v[:2]:
                print(f"  Q: {entry['prompt'][:80]}\n  A: {entry['response'][:200]}\n")

    with stage(tracking, "judge"):
        # Path A — custom panel (writes log_table + JSON artifact in this run)
        judge_result = judge_and_log(tracking, experiment="base_validation", batteries=batteries)
        print("judge summary:", judge_result["summary"])
        if _run_id and _uri:
            print(f">>> Judge metrics + table: {_uri}/#/experiments/{_exp_id}/runs/{_run_id}")
            print(f"    Metrics tab        -> judge.empathy_mean, judge.unsafe_count, etc.")
            print(f"    Artifacts tab      -> judgements/base_validation_table.json")

    with stage(tracking, "judge_mlflow_eval"):
        # Path B — MLflow GenAI Evaluation API (Feedback rows in the Evaluation tab,
        # per-row scores + rationale for each axis). Result cache means the panel
        # is called ONCE per response total across both paths above.
        # reset_judge_cache() removed — wiping it forces path B to re-call Mistral
        # for every row instead of reusing path A's cached results.
        eval_data = batteries_to_eval_dataset(batteries)
        try:
            eval_result = mlflow.genai.evaluate(data=eval_data, scorers=get_scorers())
            print(f"mlflow.genai.evaluate -> {len(eval_data)} rows scored")
            if _run_id and _uri:
                print(f">>> Evaluation tab (Feedback per row):")
                print(f"    {_uri}/#/experiments/{_exp_id}/evaluation-runs?selectedRunUuid={_run_id}")
        except Exception as exc:
            print(f"[mlflow.genai.evaluate] skipped: {exc} (need mlflow[genai] >= 3.0)")

    # scorecard from judge summary alone (no training history here)
    try:
        summary = judge_result.get("summary", {})
        sc = {}
        ov = summary.get("judge.overall_mean")
        if isinstance(ov, (int, float)): sc["judge_overall/5"] = ov / 5.0
        un, nt = summary.get("judge.unsafe_count"), summary.get("judge.n_total")
        if isinstance(un, (int, float)) and isinstance(nt, (int, float)) and nt > 0:
            sc["1-unsafe_rate"] = 1.0 - (un / nt)
        if sc:
            tracking.log_plot(plot_metric_scorecard(sc, title="base_validation scorecard"),
                              key="base_validation_scorecard")
    except Exception as exc:
        print(f"[plots] scorecard skipped: {exc}")

    tracking.log_hardware(step=1)

    # ----- FINAL "where to find everything" SUMMARY -----
    if _run_id and _uri:
        print("\n" + "="*78)
        print("  ALL ARTIFACTS LANDED. Click any link to verify in the MLflow UI.")
        print("="*78)
        print(f"  Run page         : {_uri}/#/experiments/{_exp_id}/runs/{_run_id}")
        print(f"  Metrics          : .../runs/{_run_id}  (Metrics tab — judge.*_mean, judge.unsafe_count, ...)")
        print(f"  Evaluation       : {_uri}/#/experiments/{_exp_id}/evaluation-runs?selectedRunUuid={_run_id}")
        print(f"  Artifacts        : .../runs/{_run_id}  (Artifacts tab)")
        print(f"     - judgements/base_validation_table.json   (per-row table)")
        print(f"     - judgements/<exp>.json                  (full JSON with rationales)")
        print(f"     - notebook/00_base_validation.ipynb      (live notebook)")
        print(f"     - notebook/00_base_validation.html       (rendered HTML)")
        print(f"     - responses/responses_base_validation.json")
        print(f"  Prompts (global) : {_uri}/#/prompts")
        print("="*78)

## 6. Publish this notebook (.ipynb + HTML) to MLflow

Run this cell **LAST**, AFTER the inference + judge cell above finishes. It snapshots the notebook (with all cell outputs) and uploads BOTH the `.ipynb` and an HTML render to the run's `notebook/` artifact folder.

**How it picks up the notebook (in order):**

1. **Colab** — uses the live `_message.get_ipynb` API; **no manual save needed**, the call grabs the live in-memory state directly. (Behind the scenes, Colab gives the full cell list with outputs in one shot.)
2. **VSCode** — reads the `__vsc_ipynb_file__` global VSCode injects. **Save with Ctrl/Cmd-S first** so the disk copy reflects your latest outputs (VSCode doesn't expose the live in-memory state).
3. **JupyterLab** — uses `__session__` if present, else the Jupyter REST API.
4. **Headless / papermill / unknown env** — auto-searches common roots (`/content`, cwd) for the most-recently-modified `.ipynb`.
5. **Manual override** — set `NOTEBOOK_PATH=/abs/path/to.ipynb` in env or pass `notebook_path=...`.



In [ ]:
# Publish the notebook last so it captures outputs from every cell above it.
# In Colab: save first (Ctrl-S) so the disk fallback has the latest version.


_prev_run_id = globals().get("_run_id")
if mlflow.active_run():
    _ctx2 = nullcontext()
elif _prev_run_id:
    _ctx2 = mlflow.start_run(run_id=_prev_run_id)
    print(f"[publish] re-attaching to prior run {_prev_run_id}")
else:
    _ctx2 = tracking.run(run_name=run_name + "-publish", tags=tags)
with _ctx2:
    _run = mlflow.active_run()
    _run_id = _run.info.run_id if _run else None
    _exp_id = _run.info.experiment_id if _run else None
    _uri = os.getenv("MLFLOW_TRACKING_URI", "").rstrip("/")

    # Resolve the notebook path from EVERY environment we might be in:
    #   - explicit NOTEBOOK_PATH env var
    #   - VSCode notebooks set the global __vsc_ipynb_file__
    #   - JupyterLab >=3 sets __session__
    #   - Colab uses _message.get_ipynb (handled inside log_executed_notebook)
    # If all of these fail the helper still tries disk-fallback search.
    notebook_path = (
        os.getenv("NOTEBOOK_PATH")
        or globals().get("__vsc_ipynb_file__")
        or globals().get("__session__")
    )
    try:
        result = log_executed_notebook(
            tracking,
            notebook_path=notebook_path,
            also_html=True,
            require_outputs=False,
            verbose=True,
        )
        print(f"\n[notebook] published: source={result['source']}, "
              f"had_outputs={result['had_outputs']}, html={result['html_status']}")
        if _run_id and _uri:
            print(f"\n>>> Open in MLflow UI:")
            print(f"    {_uri}/#/experiments/{_exp_id}/runs/{_run_id}/artifactPath/notebook")
    except Exception as exc:
        print(f"[notebook] publish FAILED: {exc}")
        print()
        print("Workarounds if you see this on Colab:")
        print("  1. Save the notebook (Ctrl/Cmd-S), then re-run THIS cell only.")
        print("  2. Set NOTEBOOK_PATH=/content/<your-notebook>.ipynb in env and re-run.")
        print("  3. Mount Drive: from google.colab import drive; drive.mount('/content/drive')")
        print("     then set NOTEBOOK_PATH to your Drive path.")